# TMDb Movie Enrich

Joins the daily movie export (source of truth for which movies exist) against the `asaniczka/tmdb-movies-dataset-2023-930k-movies` Kaggle dataset (source of full details).

| Step | Description |
|---|---|
| **1. Load daily** | `data/tmdb_movie_ids.csv` — 1.17M movies, 5 columns incl. dated popularity |
| **2. Load Kaggle** | Full details: genres, runtime, keywords, vote counts, etc. |
| **3. Left join** | Daily is the source of truth — only movies in the daily are kept |
| **4. Gap fetch** | Movies in daily but missing from Kaggle → fetch from `/movie/{id}` API |
| **5. Export** | Merged dataset with all columns from both sources |

## Column Reference

All columns sourced from the [TMDb Movie Details API](https://developer.themoviedb.org/reference/movie-details) via the `asaniczka/tmdb-movies-dataset-2023-930k-movies` Kaggle dataset, plus columns from the TMDb daily export.

### From the daily export

| Column | Type | Description |
|---|---|---|
| `tmdb_movie_id` | int | Unique TMDb movie identifier (`id` in the API) |
| `original_title` | string | Title in the film's original production language |
| `popularity_YYYY_MM_DD` | float | **Daily engagement score** — page views, watchlist adds, vote activity. Renamed with the snapshot date because this value changes every day. |
| `adult` | bool | `True` = adult/pornographic content |
| `video` | bool | **Media-type flag.** `True` = video artifact (featurette, bonus, event recording). `False` = standard film. Not related to trailers. |

### From the Kaggle dataset (`/movie/{id}` API)

| Column | Type | Description |
|---|---|---|
| `title` | string | Localised title (English or distribution title). Distinct from `original_title`. |
| `vote_average` | float | Mean user rating (0–10). Stable signal; not influenced by recency like `popularity`. |
| `vote_count` | float | Number of user votes. Low counts make `vote_average` unreliable. |
| `status` | string | Production/release state: `Released`, `In Production`, `Post Production`, `Planned`, `Canceled`, `Rumored` |
| `release_date` | string | `YYYY-MM-DD` theatrical release date. May be null for unreleased or undated entries. |
| `runtime` | float | Duration in minutes. Null for unreleased films. Useful filter: `> 40` removes most featurettes. |
| `budget` | float | Reported production budget in USD. Frequently 0 (not reported). |
| `revenue` | float | Reported box office revenue in USD. Frequently 0 (not reported). |
| `genres` | string | Comma-separated genre names from TMDb's fixed 19-genre taxonomy. |
| `keywords` | string | Comma-separated keyword names — the primary signal for this pipeline. |
| `original_language` | string | ISO 639-1 code of the original production language (`en`, `ja`, `fr`, …). |
| `overview` | string | Plot synopsis written by TMDb contributors. |
| `tagline` | string | Official marketing tagline. Often null. |
| `production_companies` | string | Comma-separated production company names. |
| `production_countries` | string | Comma-separated country names where the film was produced. |
| `spoken_languages` | string | Comma-separated English names of languages spoken in the film. |
| `imdb_id` | string | IMDb cross-reference ID (`tt…`). Useful for joining to external datasets. |
| `poster_path` | string | Relative path to poster image on `image.tmdb.org`. |
| `backdrop_path` | string | Relative path to backdrop image on `image.tmdb.org`. |
| `homepage` | string | Official film website URL. Frequently null. |

### Practical 'real film' filter

Combining artifact markers gives a clean boundary between narrative films and database noise:

```python
video == False          # exclude featurettes / bonus content
AND adult == False      # exclude adult content
AND status == 'Released'
AND release_date IS NOT NULL
AND runtime > 40        # exclude short films and featurettes
```

In [1]:
from pathlib import Path
import pandas as pd
import kagglehub

# ── 1. Daily export ────────────────────────────────────────────────────────
daily = pd.read_csv("data/tmdb_movie_ids.csv")
pop_col = next(c for c in daily.columns if c.startswith("popularity_"))

print(f"Daily export  : {len(daily):,} movies")
print(f"Columns       : {list(daily.columns)}")
print(f"Popularity col: {pop_col}")


Daily export  : 1,169,214 movies
Columns       : ['tmdb_movie_id', 'original_title', 'popularity_2026_03_11', 'adult', 'video']
Popularity col: popularity_2026_03_11


In [2]:
# ── 2. Kaggle dataset ──────────────────────────────────────────────────────
kaggle_path = kagglehub.dataset_download(
    "asaniczka/tmdb-movies-dataset-2023-930k-movies"
)
csv_file = next(Path(kaggle_path).glob("*.csv"))
print(f"Dataset path  : {csv_file}")

KAGGLE_DTYPES = {
    "adult":          "bool",
    "vote_average":   "float64",
    "vote_count":     "float64",
    "budget":         "float64",
    "revenue":        "float64",
    "runtime":        "float64",
}

kaggle_df = pd.read_csv(csv_file, dtype=KAGGLE_DTYPES, low_memory=False)

# Normalise id column to match daily
kaggle_df = kaggle_df.rename(columns={"id": "tmdb_movie_id"})

# Drop stale popularity — daily's dated column is the authoritative snapshot
kaggle_df = kaggle_df.drop(columns=["popularity"], errors="ignore")

print(f"Kaggle dataset: {len(kaggle_df):,} movies")
print(f"Columns ({len(kaggle_df.columns)}): {list(kaggle_df.columns)}")


Dataset path  : /Users/bobby/.cache/kagglehub/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/versions/872/TMDB_movie_dataset_v11.csv


Kaggle dataset: 1,384,645 movies
Columns (23): ['tmdb_movie_id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords']


In [3]:
# ── 3. Left join: daily is the source of truth ─────────────────────────────
merged = daily.merge(kaggle_df, on="tmdb_movie_id", how="left", suffixes=("", "_kaggle"))

# Drop any duplicate columns that came in via suffix (adult, video, original_title)
dup_cols = [c for c in merged.columns if c.endswith("_kaggle")]
merged = merged.drop(columns=dup_cols)

in_kaggle  = merged["title"].notna().sum()   # title is kaggle-only
missing    = merged["title"].isna().sum()

print(f"Merged rows       : {len(merged):,}")
print(f"Enriched from Kaggle : {in_kaggle:,} ({in_kaggle/len(merged)*100:.1f}%)")
print(f"Missing from Kaggle  : {missing:,} ({missing/len(merged)*100:.2f}%)")
print(f"\nAll columns ({len(merged.columns)}):")
print(merged.dtypes.to_string())


Merged rows       : 1,170,173
Enriched from Kaggle : 1,170,085 (100.0%)
Missing from Kaggle  : 88 (0.01%)

All columns (25):
tmdb_movie_id              int64
original_title               str
popularity_2026_03_11    float64
adult                       bool
video                       bool
title                        str
vote_average             float64
vote_count               float64
status                       str
release_date                 str
revenue                  float64
runtime                  float64
backdrop_path                str
budget                   float64
homepage                     str
imdb_id                      str
original_language            str
overview                     str
poster_path                  str
tagline                      str
genres                       str
production_companies         str
production_countries         str
spoken_languages             str
keywords                     str


In [4]:
# ── 4. Gap fetch: movies in daily missing from Kaggle ──────────────────────
import gzip, io, json, os, time
import urllib.request, urllib.error

missing_ids = merged.loc[merged["title"].isna(), "tmdb_movie_id"].tolist()
print(f"Movies to fetch from API: {len(missing_ids):,}")

TMDB_TOKEN = os.environ.get("TMDB_TOKEN", "")
RATE_LIMIT  = 40   # req / 10 s
WINDOW      = 10.0

KEEP_FIELDS = [
    "tmdb_movie_id", "title", "original_title", "original_language",
    "overview", "tagline", "status", "release_date", "runtime",
    "budget", "revenue", "vote_average", "vote_count",
    "adult", "video", "genres", "keywords",
    "production_companies", "production_countries", "spoken_languages",
    "poster_path", "backdrop_path", "homepage", "imdb_id",
]

def fetch_movie(movie_id: int, token: str) -> dict | None:
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?append_to_response=keywords"
    req = urllib.request.Request(url, headers={
        "Authorization": f"Bearer {token}",
        "accept": "application/json",
    })
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read())
            # Normalise keywords to comma-separated string (matches Kaggle format)
            kws = data.get("keywords", {}).get("keywords", [])
            data["keywords"] = ", ".join(k["name"] for k in kws)
            # Normalise genres to string
            data["genres"] = ", ".join(g["name"] for g in data.get("genres", []))
            data["production_companies"] = ", ".join(c["name"] for c in data.get("production_companies", []))
            data["production_countries"] = ", ".join(c["name"] for c in data.get("production_countries", []))
            data["spoken_languages"] = ", ".join(l["english_name"] for l in data.get("spoken_languages", []))
            data["tmdb_movie_id"] = data.pop("id")
            return {k: data.get(k) for k in KEEP_FIELDS}
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return None
        raise

if not TMDB_TOKEN:
    print("TMDB_TOKEN not set — skipping API fetch.")
    print(f"These {len(missing_ids):,} movies will have NaN for all Kaggle columns.")
    fetched_df = pd.DataFrame()
else:
    fetched = []
    times = []
    for i, mid in enumerate(missing_ids):
        now = time.monotonic()
        times = [t for t in times if now - t < WINDOW]
        if len(times) >= RATE_LIMIT:
            time.sleep(WINDOW - (now - times[0]) + 0.05)
        times.append(time.monotonic())
        result = fetch_movie(mid, TMDB_TOKEN)
        if result:
            fetched.append(result)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(missing_ids)} fetched ({len(fetched)} found)", end="\r")

    fetched_df = pd.DataFrame(fetched) if fetched else pd.DataFrame()
    print(f"\nFetched from API: {len(fetched_df):,}")


Movies to fetch from API: 88



Fetched from API: 87


In [ ]:
# ── 5. Patch gap rows into merged ──────────────────────────────────────────
if not fetched_df.empty:
    merged = merged.set_index("tmdb_movie_id")
    for _, row in fetched_df.iterrows():
        mid = row["tmdb_movie_id"]
        if mid in merged.index:
            for col, val in row.items():
                if col != "tmdb_movie_id" and col in merged.columns:
                    merged.at[mid, col] = val
    merged = merged.reset_index()

# Drop movies that couldn't be resolved (no title from Kaggle or API)
unresolvable = merged[merged["title"].isna()]
if len(unresolvable):
    print(f"Dropping {len(unresolvable):,} unresolvable movie(s): {unresolvable['tmdb_movie_id'].tolist()}")
    merged = merged[merged["title"].notna()].copy()

print(f"Final rows          : {len(merged):,}")
print(f"Fully enriched      : {merged['title'].notna().sum():,}")

OUTPUT = Path("data/tmdb_movies_enriched.csv")
merged.to_csv(OUTPUT, index=False)
print(f"Saved \u2192 {OUTPUT}  ({OUTPUT.stat().st_size / 1_048_576:.1f} MB)")
merged.head(5)

## EDA — Enriched Dataset

In [6]:
import plotly.express as px

total   = len(merged)
enriched = merged["title"].notna().sum()
missing  = total - enriched

fig = px.bar(
    x=["Enriched (Kaggle + API)", "Daily-only (no details)"],
    y=[enriched, missing],
    text_auto=True,
    color=["Enriched", "Missing"],
    color_discrete_map={"Enriched": "#1f77b4", "Missing": "#d62728"},
    title="Enrichment coverage",
    labels={"x": "", "y": "Movies"},
)
fig.update_layout(showlegend=False)
fig.show()


In [7]:
# Null rate per column — shows which fields have gaps in the Kaggle dataset
null_pct = (merged.isna().sum() / len(merged) * 100).sort_values(ascending=False)
null_df = null_pct.reset_index()
null_df.columns = ["column", "null_pct"]
null_df = null_df[null_df["null_pct"] > 0]

fig = px.bar(
    null_df,
    x="null_pct",
    y="column",
    orientation="h",
    title="Null rate per column (%)",
    labels={"column": "", "null_pct": "% null"},
    text_auto=".1f",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()
